# 05c — Evaluation & Comparison: YOLO11 vs DETR

Side-by-side comparison of:
- **Detection only**: YOLO11n vs DETR-ResNet-50 (per-class AP@50, mAP@50)
- **With trackers**: YOLO11+ByteTrack, YOLO11+BotSort, DETR+ByteTrack, DETR+BotSort
  (MOTA, IDF1, ID-Switch, per-class count metrics)

Prerequisites: `02_train_yolo.ipynb`, `03_train_rtdetr.ipynb`, `05_tracker_hyperparameter_tuning_botsort.ipynb`, `05b_tracker_hyperparameter_tuning_bytetrack.ipynb`

## ⚠️ Deprecated — Not Used in Final Project

Hyperparameter tuning of the tracker was attempted in notebooks 05, 05b, and 05c
but the results showed that the swept hyperparameter configurations performed
**worse** at track association than the baseline configuration produced by
notebook 04 (`kartector_botsort_reentry.yaml`).

**Decision**: Tracker hyperparameter tuning has been abandoned. All downstream
notebooks (06, 07, 08) use the **baseline arch config** from notebook 04 directly.

These notebooks are retained for reference only and are not part of the active pipeline.


## 0 — Configuration

In [ ]:
from pathlib import Path
import json, numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

REPO_ROOT     = Path('/home1/hendersonj6179@cgu.edu')
YOLO_RUNS     = (REPO_ROOT / 'runs/yolo26').resolve()
RTDETR_RUNS   = (REPO_ROOT / 'runs/rtdetr').resolve()
# Tracker output dirs written by 05 / 05b
TRACKER_YOLO_BS  = (REPO_ROOT / 'runs/trackers').resolve()
TRACKER_YOLO_BT  = (REPO_ROOT / 'runs/trackers_bytetrack').resolve()
TRACKER_RTDETR_BS = (REPO_ROOT / 'runs/trackers_rtdetr_botsort').resolve()
TRACKER_RTDETR_BT = (REPO_ROOT / 'runs/trackers_rtdetr_bytetrack').resolve()
YOLO_WEIGHTS  = YOLO_RUNS / 'final' / 'weights' / 'best.pt'
RTDETR_WEIGHTS = RTDETR_RUNS / 'final' / 'weights' / 'best.pt'
# Final workflow configs produced by 05 and 05b
CONFIGS = {
    'botsort_yolo':    REPO_ROOT / 'configs' / 'kartector_botsort_yolo_final.json',
    'bytetrack_yolo':  REPO_ROOT / 'configs' / 'kartector_bytetrack_yolo_final.json',
    'botsort_rtdetr':  REPO_ROOT / 'configs' / 'kartector_botsort_rtdetr_final.json',
    'bytetrack_rtdetr':REPO_ROOT / 'configs' / 'kartector_bytetrack_rtdetr_final.json',
}
def load_tracker_cfg(name):
    p = CONFIGS[name]
    if not p.exists():
        print(f'  WARNING: {p} not found — run 05/05b first')
        return None
    return json.loads(p.read_text())
DATA_YAML     = REPO_ROOT / 'data/yolo_dataset_cv_reduced/full_train/data.yaml'
VAL_IMG_DIR   = REPO_ROOT / 'data/yolo_dataset_cv_reduced/full_train/images/val'
VAL_LBL_DIR   = REPO_ROOT / 'data/yolo_dataset_cv_reduced/full_train/labels/val'
COMPARE_DIR   = (REPO_ROOT / 'runs/comparison').resolve()
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

with open(REPO_ROOT / 'data/splits/splits.json') as f:
    splits = json.load(f)
CLASSES = splits['classes']
N_CLASSES = len(CLASSES)

TRACKERS = ['bytetrack', 'botsort']
COLORS = {'yolo_bytetrack':'#1f77b4','yolo_botsort':'#aec7e8',
          'detr_bytetrack':'#d62728','detr_botsort':'#f7b6d2'}

print('Config OK')
print(f'  YOLO weights : {YOLO_WEIGHTS}')
print(f'  DETR dir     : {DETR_DIR}')
print(f'  Classes      : {CLASSES}')

## 1 — Detection Comparison (Image-Level AP)

Evaluate both models on the same val set and report per-class AP@50 and mAP@50.

In [ ]:
# ── YOLO detection AP ────────────────────────────────────────────────────
yolo_ap = {}; yolo_map50 = None
if YOLO_WEIGHTS.exists():
    from ultralytics import YOLO
    m = YOLO(str(YOLO_WEIGHTS)).val(data=str(DATA_YAML.resolve()),
                                     imgsz=320, device=DEVICE, verbose=False)
    ap_idx  = m.box.ap_class_index.tolist() if hasattr(m.box.ap_class_index,'tolist') else list(m.box.ap_class_index)
    ap_vals = m.box.ap50.tolist()           if hasattr(m.box.ap50,'tolist')           else list(m.box.ap50)
    yolo_ap = {CLASSES[i]: v for i,v in zip(ap_idx, ap_vals)}
    yolo_map50 = m.box.map50
    print(f'YOLO mAP@50 = {yolo_map50:.4f}')
    for cls, ap in yolo_ap.items(): print(f'  {cls:20s}: {ap:.4f}')
else:
    print(f'YOLO weights not found: {YOLO_WEIGHTS}')

In [ ]:
# ── DETR detection AP ────────────────────────────────────────────────────
detr_ap = {}; detr_map50 = None
detr_ap_path = DETR_RUNS / 'final_ap.json'
if detr_ap_path.exists():
    # Load pre-computed from notebook 04 if available
    with open(detr_ap_path) as f:
        saved = json.load(f)
    detr_ap   = saved['ap_per_class']
    detr_map50= saved['map50']
    print(f'DETR mAP@50 = {detr_map50:.4f} (loaded from 04_train_detr)')
elif DETR_DIR.exists():
    # Re-compute if needed
    from transformers import DetrForObjectDetection, DetrImageProcessor
    from pycocotools.coco import COCO; from pycocotools.cocoeval import COCOeval
    from PIL import Image; import tempfile
    processor = DetrImageProcessor.from_pretrained(str(DETR_DIR))
    model = DetrForObjectDetection.from_pretrained(str(DETR_DIR)).to(DEVICE); model.eval()
    img_paths = sorted([p for p in VAL_IMG_DIR.iterdir() if p.suffix.lower() in ('.jpg','.jpeg','.png')])
    coco_gt = {'images':[],'annotations':[],'categories':[{'id':i,'name':c} for i,c in enumerate(CLASSES)]}
    coco_dt = []; ann_id=0
    for img_id, img_path in enumerate(img_paths):
        image = Image.open(img_path).convert('RGB'); W,H=image.size
        coco_gt['images'].append({'id':img_id,'width':W,'height':H})
        lbl_path = VAL_LBL_DIR/(img_path.stem+'.txt')
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().splitlines():
                parts=line.split()
                if len(parts)==5:
                    cls,cx,cy,w,h=int(parts[0]),*map(float,parts[1:])
                    x1=(cx-w/2)*W; y1=(cy-h/2)*H
                    coco_gt['annotations'].append({'id':ann_id,'image_id':img_id,'category_id':cls,
                        'bbox':[x1,y1,w*W,h*H],'area':w*W*h*H,'iscrowd':0})
                    ann_id+=1
        inputs=processor(images=image,return_tensors='pt').to(DEVICE)
        with torch.no_grad(): outputs=model(**inputs)
        results=processor.post_process_object_detection(outputs,threshold=0.05,target_sizes=torch.tensor([[H,W]]).to(DEVICE))[0]
        for score,label_id,box in zip(results['scores'],results['labels'],results['boxes']):
            x1,y1,x2,y2=box.tolist()
            coco_dt.append({'image_id':img_id,'category_id':int(label_id),'bbox':[x1,y1,x2-x1,y2-y1],'score':float(score)})
    with tempfile.NamedTemporaryFile('w',suffix='.json',delete=False) as f2:
        json.dump(coco_gt,f2); gt_path=f2.name
    coco_gt_api=COCO(gt_path)
    if coco_dt:
        coco_dt_api=coco_gt_api.loadRes(coco_dt)
        ev=COCOeval(coco_gt_api,coco_dt_api,'bbox'); ev.evaluate(); ev.accumulate(); ev.summarize()
        for i,cls in enumerate(CLASSES):
            ev.params.catIds=[i]; ev.evaluate(); ev.accumulate()
            detr_ap[cls]=ev.stats[1]
        detr_map50=float(np.mean(list(detr_ap.values())))
    Path(gt_path).unlink(missing_ok=True)
    print(f'DETR mAP@50 = {detr_map50:.4f}')
else:
    print(f'DETR model not found: {DETR_DIR}')

In [ ]:
# ── Side-by-side AP bar chart ─────────────────────────────────────────────
if yolo_ap and detr_ap:
    x = np.arange(N_CLASSES); width=0.35
    yolo_vals = [yolo_ap.get(c,0) for c in CLASSES]
    detr_vals = [detr_ap.get(c,0) for c in CLASSES]
    fig,ax=plt.subplots(figsize=(12,5))
    b1=ax.bar(x-width/2,yolo_vals,width,label=f'YOLO11 mAP@50={yolo_map50:.3f}',color='#1f77b4',alpha=0.85)
    b2=ax.bar(x+width/2,detr_vals,width,label=f'DETR    mAP@50={detr_map50:.3f}',color='#d62728',alpha=0.85)
    for bar,val in [(b,v) for bars,vals in [(b1,yolo_vals),(b2,detr_vals)] for b,v in zip(bars,vals)]:
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,f'{val:.2f}',ha='center',fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(CLASSES,rotation=30,ha='right')
    ax.set_ylabel('AP@50'); ax.set_title('Per-Class AP@50: YOLO11 vs DETR',fontweight='bold')
    ax.set_ylim(0,1.15); ax.legend(); ax.grid(axis='y',alpha=0.3)
    plt.tight_layout()
    out = COMPARE_DIR/'detection_ap_comparison.png'
    plt.savefig(out,dpi=150,bbox_inches='tight'); plt.show(); print(f'Saved: {out}')

# Summary table
det_rows = []
for c in CLASSES:
    det_rows.append({'class':c,'YOLO_AP50':round(yolo_ap.get(c,float('nan')),4),'DETR_AP50':round(detr_ap.get(c,float('nan')),4)})
det_rows.append({'class':'mAP@50','YOLO_AP50':round(yolo_map50 or float('nan'),4),'DETR_AP50':round(detr_map50 or float('nan'),4)})
df_det = pd.DataFrame(det_rows)
print(df_det.to_string(index=False))
df_det.to_csv(COMPARE_DIR/'detection_ap.csv',index=False)

## 2 — Tracking Metrics Comparison (MOTA / IDF1 / ID-Switch)

Load saved `test_metrics.json` from notebooks 03 and 05.

In [ ]:
tracking_results = {}

# Load all four final configs produced by notebooks 05 and 05b
for cfg_name, cfg_path in CONFIGS.items():
    if not cfg_path.exists():
        print(f"  SKIP {cfg_name}: {cfg_path} not found")
        continue
    cfg = json.loads(cfg_path.read_text())
    backbone = cfg.get("backbone", "yolo")
    tracker  = cfg.get("tracker", cfg_name.split("_")[0])
    # Determine tracker runs dir
    runs_map = {
        ("botsort",    "yolo"):   TRACKER_YOLO_BS,
        ("bytetrack",  "yolo"):   TRACKER_YOLO_BT,
        ("botsort",    "rtdetr"): TRACKER_RTDETR_BS,
        ("bytetrack",  "rtdetr"): TRACKER_RTDETR_BT,
    }
    runs_dir = runs_map.get((tracker, backbone))
    metrics_path = runs_dir / "test_metrics.json" if runs_dir else None
    if metrics_path and metrics_path.exists():
        m = json.loads(metrics_path.read_text())
        key = cfg_name
        # test_metrics.json may contain {tracker_name: metrics} or flat metrics
        if isinstance(list(m.values())[0], dict):
            metrics = list(m.values())[0]
        else:
            metrics = m
        tracking_results[key] = metrics
        print(f"  {key:28s}: MOTA={metrics.get('MOTA')}  "
              f"IDF1={metrics.get('IDF1')}  ID_Sw={metrics.get('ID_Sw')}")
    else:
        print(f"  SKIP {cfg_name}: metrics not found at {metrics_path}")


In [ ]:
# ── Grouped bar chart: MOTA / IDF1 / ID_Sw ──────────────────────────────
if tracking_results:
    labels = list(tracking_results.keys())
    bar_colors = [COLORS.get(k,'#999999') for k in labels]
    fig,axes=plt.subplots(1,3,figsize=(16,5))
    for ax,metric in zip(axes,['MOTA','IDF1','ID_Sw']):
        vals=[tracking_results[k].get(metric) or 0 for k in labels]
        bars=ax.bar(labels,vals,color=bar_colors,alpha=0.85)
        ax.set_title(metric,fontsize=13)
        ax.set_ylim(0,max(vals)*1.3 if max(vals)>0 else 1)
        for bar,val in zip(bars,vals):
            ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.005,
                    f'{val:.3f}' if metric!='ID_Sw' else str(int(val)),
                    ha='center',va='bottom',fontsize=9)
        ax.set_xticklabels(labels,rotation=25,ha='right',fontsize=9)
        ax.grid(axis='y',alpha=0.3)
    plt.suptitle('Tracking Metrics: YOLO11 vs DETR  (ByteTrack & BotSort)',
                 fontweight='bold',fontsize=14)
    plt.tight_layout()
    out=COMPARE_DIR/'tracking_metrics_comparison.png'
    plt.savefig(out,dpi=150,bbox_inches='tight'); plt.show(); print(f'Saved: {out}')

    df_track=pd.DataFrame([{'model':k,**v} for k,v in tracking_results.items()])
    print(df_track.round(4).to_string(index=False))
    df_track.to_csv(COMPARE_DIR/'tracking_metrics.csv',index=False)

## 3 — Per-Class Count Metrics (CA, CE, ACE)

Compare count accuracy across all four model+tracker combinations.

In [ ]:
count_results = {}

for prefix, tracker_dir in [('yolo', TRACKER_YOLO_BS), ('detr', TRACKER_RTDETR_BS)]:
    p = tracker_dir / 'count_metrics.json'
    if not p.exists(): print(f'Not found: {p}'); continue
    with open(p) as f: cm = json.load(f)
    for tname, rows in cm.items():
        count_results[f'{prefix}_{tname}'] = pd.DataFrame(rows)

if count_results:
    metrics_to_plot = ['CA','CE','ACE']
    labels = list(count_results.keys())
    bar_colors = [COLORS.get(k,'#999999') for k in labels]
    class_labels = CLASSES
    for metric in metrics_to_plot:
        fig,ax=plt.subplots(figsize=(14,5))
        x=np.arange(len(class_labels)); width=0.8/len(labels)
        for ti,(k,df) in enumerate(count_results.items()):
            vals=[df.loc[df['class']==c,metric].values[0] if c in df['class'].values else 0 for c in class_labels]
            offset=(ti-len(labels)/2+0.5)*width
            ax.bar(x+offset,vals,width,label=k,color=bar_colors[ti],alpha=0.85)
        if metric=='CE': ax.axhline(0,color='black',lw=0.8,ls='--')
        ax.set_xticks(x); ax.set_xticklabels(class_labels,rotation=30,ha='right')
        ax.set_title(f'Per-Class {metric}',fontweight='bold'); ax.set_ylabel(metric)
        ax.legend(fontsize=8); ax.grid(axis='y',alpha=0.3)
        plt.tight_layout()
        out=COMPARE_DIR/f'count_{metric.lower()}_comparison.png'
        plt.savefig(out,dpi=150,bbox_inches='tight'); plt.show(); print(f'Saved: {out}')

## 4 — Summary Table

In [ ]:
print('\n' + '='*70)
print('DETECTION COMPARISON')
print('='*70)
if 'df_det' in dir(): print(df_det.to_string(index=False))

print('\n' + '='*70)
print('TRACKING COMPARISON')
print('='*70)
if tracking_results:
    df_summary = pd.DataFrame([
        {'model': k,
         'MOTA':  round(v.get('MOTA') or float('nan'),4),
         'IDF1':  round(v.get('IDF1') or float('nan'),4),
         'ID_Sw': v.get('ID_Sw') or 0}
        for k,v in tracking_results.items()
    ])
    print(df_summary.to_string(index=False))
    df_summary.to_csv(COMPARE_DIR/'summary.csv',index=False)
    # Highlight best in each column
    best_mota = df_summary.loc[df_summary['MOTA'].idxmax(),'model']
    best_idf1 = df_summary.loc[df_summary['IDF1'].idxmax(),'model']
    best_ids  = df_summary.loc[df_summary['ID_Sw'].idxmin(),'model']
    print(f'\nBest MOTA  : {best_mota}')
    print(f'Best IDF1  : {best_idf1}')
    print(f'Fewest ID-Sw: {best_ids}')

## 5 — Radar Chart (Normalised Metrics)

In [ ]:
import matplotlib
if tracking_results and 'df_summary' in dir():
    metrics_r = ['MOTA','IDF1']
    # Normalise ID_Sw inversely: norm = 1 - (id_sw / max_id_sw)
    max_ids = max(v.get('ID_Sw') or 0 for v in tracking_results.values()) or 1
    def norm_ids(v): return 1 - (v.get('ID_Sw') or 0)/max_ids
    # Range-normalise MOTA and IDF1 across models
    def norm_metric(key):
        vals=[v.get(key) or 0 for v in tracking_results.values()]
        mn,mx=min(vals),max(vals)
        if mx==mn: return {k:0.5 for k in tracking_results}
        return {k:(v.get(key) or 0-mn)/(mx-mn) for k,v in tracking_results.items()}
    nm=norm_metric('MOTA'); ni=norm_metric('IDF1')
    radar_labels=['MOTA (norm)','IDF1 (norm)','1-ID_Sw (norm)']
    n_angles=len(radar_labels)
    angles=[a/float(n_angles)*2*np.pi for a in range(n_angles)]+[0]
    fig,ax=plt.subplots(figsize=(7,7),subplot_kw={'polar':True})
    for i,(k,v) in enumerate(tracking_results.items()):
        vals=[nm.get(k,0),ni.get(k,0),norm_ids(v)]
        vals+=vals[:1]
        color=COLORS.get(k,'#999999')
        ax.plot(angles,vals,color=color,lw=2,label=k); ax.fill(angles,vals,color=color,alpha=0.1)
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(radar_labels,fontsize=10)
    ax.set_ylim(0,1); ax.set_title('Radar — Normalised Tracking Metrics',fontweight='bold',pad=20)
    ax.legend(loc='upper right',bbox_to_anchor=(1.3,1.1),fontsize=9)
    plt.tight_layout()
    out=COMPARE_DIR/'radar_chart.png'
    plt.savefig(out,dpi=150,bbox_inches='tight'); plt.show(); print(f'Saved: {out}')